In [1]:
import gc
import glob

import numpy as np
import pandas as pd

from pyliftover import LiftOver

import matplotlib.pyplot as plt
import seaborn as sns

import anndata as ad
import scanpy as sc

%matplotlib inline

In [2]:
## Read In Data ##

adata = sc.read_h5ad('/mnt/sdb/scz_meta_analysis_processed/anndata_objs/integrated_adata.h5ad')

# read in cluster identites from hierarchical_clustering_scVI  
cluster_identities = pd.read_csv('/mnt/sdb/scz_meta_analysis_processed/dge_signatures/iterative_clustering_table_plus_missing.csv', index_col=0)
del adata.obs['class']
adata.obs = adata.obs.merge(cluster_identities, left_index=True, right_index=True, how='left')

In [3]:
# Live or Fixed - all are live except for some from Sebastian 

# generalize sebastians procedure column to the rest
adata.obs['procedure'].fillna('live', inplace=True)
adata.obs.rename({'procedure':'Live_Or_Fixed'}, inplace=True, axis=1)

In [4]:
## Ambiguous Columns ##

# Unaffected Twin # Sawada
# Affected Twin # Sawada
# Criteria # Sawada, what critiera used to diagnosis SCZ e.g. DSM IV or MINI
# doublet_logLikRatio # Shin , part of vireo data for Shin
# best_doublet # Shin , part of vireo data for Shin
# best_singlet # Shin , part of vireo data for Shin
# Organoid_Origin # Walsh, tells us whether it came from a dorsal vs ventral organoid or assembloid 
# Columbia ID # Sebastian, will give you the QR ID for the Rutger's Lines present in Rao and Notaras well be coverting

In [5]:
# Clinical Notes from Notaras Needs to be Integrated #

adata.obs['Diagnosis'] = adata.obs['Diagnosis'].astype('str')
adata.obs['Clinical Notes'] = adata.obs['Clinical Notes'].astype('str')
adata.obs.loc[adata.obs['Diagnosis'].isnull(), 'Diagnosis'] = adata.obs['Clinical Notes']
del adata.obs['Clinical Notes']

In [6]:
# Regional Differentiation # - dorsal vs ventral telecephalon

# cell_type from fernando is a organoid differentiation and should be merged with Differentiation from Walsh
# most datasets missing Differentiation but all are Cortical except parts of Fernando and Walsh

adata.obs['cell_type'] = adata.obs['cell_type'].astype('str')
adata.obs['Differentiation'] = adata.obs['Differentiation'].astype('str')
adata.obs.loc[adata.obs['Differentiation'] == 'nan', 'Differentiation'] = adata.obs['cell_type']
adata.obs['Differentiation'] = adata.obs['Differentiation'].str.replace('hiPSC-derived-hCS', 'Cortical')
adata.obs['Differentiation'] = adata.obs['Differentiation'].str.replace('hiPSC-derived-hSS', 'Ventral')
adata.obs['Differentiation'] = adata.obs['Differentiation'].str.replace('Cortical Organoid', 'Cortical')
adata.obs.loc[adata.obs['Differentiation'] == 'nan', 'Differentiation'] = 'Cortical' # impute remaining as cortical
del adata.obs['cell_type']

In [7]:
# Homogenize Reprogramming Method #

# "Method of reprogramming" is Khan & "Derived by" is Shin #

adata.obs['Method of reprogramming'] = adata.obs['Method of reprogramming'].astype('str')
adata.obs['Derived by'] = adata.obs['Derived by'].astype('str')
adata.obs.loc[adata.obs['Method of reprogramming'].isnull(), 'Method of reprogramming'] = adata.obs['Derived by'].astype('str')
adata.obs.rename({'Method of reprogramming':'Reprogramming_Method'}, axis=1, inplace=True)
adata.obs['Reprogramming_Method'] = adata.obs['Reprogramming_Method'].str.replace('Episomal, Y4 mixture', 'Episomal')
del adata.obs['Derived by']

In [8]:
# Homonogenize Ancestry/Ethncity from Notaras, Shin, Fernando

# Fernando uses European and Non European only. Convert to Caucasian.
# Notaras donors are all is all White (Caucasian)
# Shin donors are a mixture of Caucasian/Japanese/Unknown/Ad-mixed American

adata.obs['Ethnicity'] = adata.obs['Ethnicity'].astype('str')
adata.obs.loc[((adata.obs['Ethnicity'].isnull()) | (adata.obs['Ethnicity'] == 'nan')), 'Ethnicity'] = adata.obs['Ancestry']
adata.obs['Ethnicity'] = adata.obs['Ethnicity'].str.replace('European', 'Caucasian')
adata.obs['Ethnicity'] = adata.obs['Ethnicity'].str.replace('White', 'Caucasian')
del adata.obs['Ancestry']
adata.obs.rename({'Ethnicity':'Ancestry'}, inplace=True, axis=1)

In [9]:
# Homonogenize Age of Onset (First Break) #

# Age of Onset - Notaras
# Onset - Sawada
# Age of onset (y) - Fernando

# rework 20s as 20 as closest estimate and <8 to 8

adata.obs['Age of Onset'] = adata.obs['Age of Onset'].astype('str')
adata.obs.loc[adata.obs['Age of Onset'] == 'nan', 'Age of Onset'] = adata.obs['Onset']
adata.obs.loc[adata.obs['Age of Onset'].isnull(), 'Age of Onset'] = adata.obs['Age of onset (y)']
adata.obs['Age of Onset'] = adata.obs['Age of Onset'].str.replace('<','').str.replace('s', '') 
adata.obs['Age of Onset'] = adata.obs['Age of Onset'].replace('Unknown', np.nan)
adata.obs['Age of Onset'] = adata.obs['Age of Onset'].replace('N/A', np.nan)
adata.obs['Age of Onset'] = adata.obs['Age of Onset'].astype('float')
del adata.obs['Age of onset (y)']
del adata.obs['Onset']

In [10]:
# Homogenize Cell Type of Origin

# Parental Cells for iPSC reprogramming substrate - Sawada
# Cell Type of Origin  reprogramming substrate - Shin 

adata.obs['Parental Cells for iPSC'] = adata.obs['Parental Cells for iPSC'].astype('str')
adata.obs.loc[(adata.obs['Parental Cells for iPSC'] == 'nan'), 'Parental Cells for iPSC'] = adata.obs['Cell Type of Origin']
adata.obs['Parental Cells for iPSC'].replace('Unknown', np.nan, inplace=True)
del adata.obs['Cell Type of Origin']

In [11]:
# Homogenize Imputed Sex 

# Notaras had Imputed_Male and Imputed_Female and Male.
adata.obs['Imputed_Sex'] = adata.obs['Imputed_Sex'].str.contains('Imputed', na=False).map({True: 'Imputed', False: 'Not'})

In [12]:
## Save From Dreamlet ##

adata.obs['Sample Name'] = adata.obs['Sample Name'].astype('str')

adata.obs['Run_Donor'] = adata.obs['Run'].astype('str') + '_' + adata.obs['Donor'].astype('str')
adata.obs['Run_Donor_Sample'] = adata.obs['Run_Donor'] + '_' + adata.obs['Sample Name']
adata.X = adata.layers['raw_counts']
del adata.layers

adata.write_h5ad('/mnt/sdb/scz_meta_analysis_processed/anndata_objs/one_versus_all_for_dreamlet.h5ad')